# 05. 운전자 수익성 분석

DRIVER_ID별 매출·건수·거리·실차율·시간당매출을 청크 누적으로 집계하고,
상위/하위 10% 운전자 비교, 플랫폼 수수료, 등급 분류를 분석한다.

> 운임 주의: PAY_AMT(결제금액) 기준. 명세서상 운임은 CARD_RIDE_FARE+CASH_RIDE_FARE이나
> 본 분석은 실제 결제액(PAY_AMT)을 매출 프록시로 사용한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
import seaborn as sns

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. 청크 집계: 운전자별 / 시간대·요일별

In [ ]:
D012_PATH = './DC_TBYXD012.csv'
usecols = ['RIDE_DTIME','ALIGHT_DTIME','PAY_AMT','RIDE_DIST','VACNTV_DIST','DRIVER_ID','PLTF_FEE_AMT']
dtypes  = {'RIDE_DTIME': str, 'ALIGHT_DTIME': str, 'PAY_AMT':'float64',
           'RIDE_DIST':'float64','VACNTV_DIST':'float64','DRIVER_ID': str,'PLTF_FEE_AMT':'float64'}

drv_parts, hr_parts = [], []
# 상위/하위 비교용: 운전자별 시간대·지역 분포는 등급 확정 후 2차 패스
total = 0
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    ad = pd.to_datetime(chunk['ALIGHT_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna() & ad.notna()
    chunk, rd, ad = chunk[m].copy(), rd[m], ad[m]
    dur = (ad - rd).dt.total_seconds() / 60
    chunk['dur'] = dur.where((dur > 0) & (dur <= 180))
    chunk['hour'] = rd.dt.hour
    chunk['weekday'] = rd.dt.dayofweek
    total += len(chunk)

    drv_parts.append(chunk.groupby('DRIVER_ID').agg(
        revenue=('PAY_AMT','sum'), trips=('PAY_AMT','size'),
        ride_dist=('RIDE_DIST','sum'), vac_dist=('VACNTV_DIST','sum'),
        dur_sum=('dur','sum'), fee=('PLTF_FEE_AMT','sum')).reset_index())
    hr_parts.append(chunk.groupby(['weekday','hour']).agg(
        revenue=('PAY_AMT','sum'), trips=('PAY_AMT','size'), dur_sum=('dur','sum')).reset_index())
    del chunk, rd, ad, dur; gc.collect()

driver = drv_parts[0].iloc[0:0]
driver = pd.concat(drv_parts).groupby('DRIVER_ID').sum().reset_index()
driver['avg_fare'] = driver['revenue'] / driver['trips']
driver['occupied_rate'] = driver['ride_dist'] / (driver['ride_dist'] + driver['vac_dist']).replace(0, np.nan)
driver['rph'] = driver['revenue'] / (driver['dur_sum'] / 60).replace(0, np.nan)
hourly = pd.concat(hr_parts).groupby(['weekday','hour']).sum().reset_index()
hourly['rph'] = hourly['revenue'] / (hourly['dur_sum'] / 60).replace(0, np.nan)
del drv_parts, hr_parts; gc.collect()

print(f"전체 {total:,}건, 운전자 {len(driver):,}명")
mem_usage('after load')
driver.head()

## 2. 실차율 분포

In [ ]:
occ = driver['occupied_rate'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(occ, bins=30, color='steelblue', edgecolor='black', lw=0.5)
axes[0].axvline(occ.mean(), color='red', ls='--', label=f'평균 {occ.mean():.1%}')
axes[0].set_title('운전자별 실차율 분포'); axes[0].set_xlabel('실차율'); axes[0].legend()
axes[1].scatter(driver['occupied_rate'], driver['revenue'], alpha=0.4, s=12, color='steelblue')
axes[1].set_title('실차율 vs 총매출'); axes[1].set_xlabel('실차율'); axes[1].set_ylabel('총매출(원)')
plt.tight_layout(); plt.show()
print(f"평균 실차율 {occ.mean():.2%}, 중앙값 {occ.median():.2%}")

## 3. 시간당 매출 분포

In [ ]:
rph = driver['rph'].dropna()
q1, q99 = rph.quantile(0.01), rph.quantile(0.99)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(rph[(rph>=q1)&(rph<=q99)], bins=40, color='coral', edgecolor='black', lw=0.5)
ax.axvline(rph.median(), color='blue', ls='--', label=f'중앙값 {rph.median():,.0f}원')
ax.set_title('운전자별 시간당 매출 분포'); ax.set_xlabel('시간당 매출(원)'); ax.legend()
plt.tight_layout(); plt.show()
print(f"시간당 매출 평균 {rph.mean():,.0f}원, 중앙값 {rph.median():,.0f}원")

## 4. 시간대/요일별 시간당 매출 히트맵

In [ ]:
dkr = {0:'월',1:'화',2:'수',3:'목',4:'금',5:'토',6:'일'}
piv = hourly.pivot_table(index='weekday', columns='hour', values='rph')
piv.index = [dkr[i] for i in piv.index]
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(piv, cmap='YlOrRd', ax=ax, annot=True, fmt=',.0f', linewidths=0.5, annot_kws={'fontsize':7})
ax.set_title('시간대/요일별 시간당 매출(원)', fontweight='bold'); ax.set_xlabel('시간'); ax.set_ylabel('요일')
plt.tight_layout(); plt.show()

## 5. 상위 10% vs 하위 10% 운전자

In [ ]:
q90, q10 = driver['revenue'].quantile(0.9), driver['revenue'].quantile(0.1)
top10 = driver[driver['revenue'] >= q90]
bot10 = driver[driver['revenue'] <= q10]
compare = pd.DataFrame({
    '상위10%': [f"{top10['revenue'].mean():,.0f}", f"{top10['trips'].mean():.1f}",
               f"{top10['avg_fare'].mean():,.0f}", f"{top10['occupied_rate'].mean():.2%}",
               f"{top10['rph'].mean():,.0f}"],
    '하위10%': [f"{bot10['revenue'].mean():,.0f}", f"{bot10['trips'].mean():.1f}",
               f"{bot10['avg_fare'].mean():,.0f}", f"{bot10['occupied_rate'].mean():.2%}",
               f"{bot10['rph'].mean():,.0f}"],
}, index=['평균 총매출','평균 건수','평균 요금','평균 실차율','시간당 매출'])
print('=== 상위10% vs 하위10% ==='); compare

## 6. 플랫폼 수수료 영향

In [ ]:
driver['fee_rate'] = (driver['fee'] / driver['revenue'].replace(0, np.nan))
driver['net_revenue'] = driver['revenue'] - driver['fee']
driver['net_rph'] = driver['net_revenue'] / (driver['dur_sum']/60).replace(0, np.nan)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(driver['fee_rate'].dropna(), bins=30, color='steelblue', edgecolor='black', lw=0.5)
axes[0].set_title('수수료율 분포'); axes[0].set_xlabel('수수료/매출')
axes[1].scatter(driver['revenue'], driver['fee'], alpha=0.4, s=12, color='steelblue')
axes[1].set_title('총매출 vs 수수료'); axes[1].set_xlabel('총매출'); axes[1].set_ylabel('수수료')
axes[2].scatter(driver['fee_rate'], driver['net_rph'], alpha=0.4, s=12, color='coral')
axes[2].set_title('수수료율 vs 시간당 순매출'); axes[2].set_xlabel('수수료율'); axes[2].set_ylabel('시간당 순매출')
plt.tight_layout(); plt.show()
print(f"평균 수수료율 {driver['fee_rate'].mean():.2%}, 수수료 총액 {driver['fee'].sum():,.0f}원")

## 7. 운전자 등급 분류 (A/B/C/D)

In [ ]:
d = driver.dropna(subset=['occupied_rate']).copy()
rmin, rmax = d['revenue'].min(), d['revenue'].max()
omin, omax = d['occupied_rate'].min(), d['occupied_rate'].max()
d['rev_norm'] = (d['revenue'] - rmin) / (rmax - rmin + 1e-9)
d['occ_norm'] = (d['occupied_rate'] - omin) / (omax - omin + 1e-9)
d['score'] = d['rev_norm'] * d['occ_norm']
d['grade'] = pd.qcut(d['score'], 4, labels=['D','C','B','A'])
gstats = d.groupby('grade', observed=True).agg(
    운전자수=('DRIVER_ID','count'), 평균매출=('revenue','mean'),
    평균실차율=('occupied_rate','mean'), 평균시간당매출=('rph','mean'),
    평균수수료율=('fee_rate','mean')).round(2)
print('=== 등급별 통계 ==='); gstats

In [ ]:
gc_ = {'A':'#2ecc71','B':'#3498db','C':'#f39c12','D':'#e74c3c'}
counts = d['grade'].value_counts().sort_index(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(counts.index.astype(str), counts.values, color=[gc_[g] for g in counts.index], edgecolor='black', lw=0.5)
axes[0].set_title('운전자 등급 분포'); axes[0].set_ylabel('운전자 수')
box = [d[d['grade']==g]['revenue'].values for g in ['A','B','C','D']]
bp = axes[1].boxplot(box, labels=['A','B','C','D'], patch_artist=True, showfliers=False)
for patch, g in zip(bp['boxes'], ['A','B','C','D']): patch.set_facecolor(gc_[g]); patch.set_alpha(0.6)
axes[1].set_title('등급별 총매출 분포'); axes[1].set_ylabel('총매출(원)')
plt.tight_layout(); plt.show()

## 8. 요약

In [ ]:
print('=== 운전자 수익성 요약 ===')
for k, v in {
    '총 운전자 수': f"{len(driver):,}",
    '총 통행 건수': f"{driver['trips'].sum():,.0f}",
    '전체 매출': f"{driver['revenue'].sum():,.0f}원",
    '운전자당 평균매출': f"{driver['revenue'].mean():,.0f}원",
    '평균 실차율': f"{driver['occupied_rate'].mean():.2%}",
    '시간당 매출 중앙값': f"{driver['rph'].median():,.0f}원",
    '평균 수수료율': f"{driver['fee_rate'].mean():.2%}",
    'A등급 수': f"{(d['grade']=='A').sum():,}",
    'D등급 수': f"{(d['grade']=='D').sum():,}",
}.items(): print(f'  {k}: {v}')